# Chapter 8 Companion Notebook: Market Risk Management

This notebook reproduces every worked numerical example from Chapter 8 of *AI in Finance*: parametric/historical/Monte Carlo VaR and Expected Shortfall, matrix-based portfolio VaR and Component VaR, GARCH-based and filtered-historical-simulation VaR, VaR backtesting (Kupiec's test, the Basel traffic-light framework, and ES backtesting), the square-root-of-time scaling rule and its limits under autocorrelated returns, liquidity-adjusted VaR and the NSFR, delta-normal and delta-gamma VaR for options, stress testing, correlation stress testing, reverse stress testing and a VaR limit-breach example, and a head-to-head test of GARCH(1,1) versus gradient boosting for volatility forecasting on simulated asymmetric (leverage-effect) data.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Value at Risk: parametric and historical methods (Section 8.2.2)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

returns = np.array([0.0006, 0.0042, -0.0027, -0.0101, -0.0049, -0.0113, 0.0013, 0.0167,
                     -0.0053, -0.0068, 0.0065, 0.0049, 0.0019, -0.0106, 0.0002, 0.0089,
                     -0.0155, -0.0049, -0.0222, -0.0149])
V = 10_000_000

mu = returns.mean()
sigma = returns.std(ddof=1)
print(f"mean = {mu:.4%}, std = {sigma:.4%}")

z95, z99 = norm.ppf(0.95), norm.ppf(0.99)
VaR95_param = -(mu - z95 * sigma) * V
VaR99_param = -(mu - z99 * sigma) * V
print(f"Parametric VaR95: ${VaR95_param:,.0f}")
print(f"Parametric VaR99: ${VaR99_param:,.0f}")

sorted_returns = np.sort(returns)
idx95 = int(np.floor(0.05 * len(returns)))
VaR95_hist = -sorted_returns[idx95] * V
print(f"\nSorted returns: {sorted_returns}")
print(f"Historical VaR95 (2nd-worst return): ${VaR95_hist:,.0f}")

mean = -0.3200%, std = 0.9374%
Parametric VaR95: $186,195
Parametric VaR99: $250,081

Sorted returns: [-0.0222 -0.0155 -0.0149 -0.0113 -0.0106 -0.0101 -0.0068 -0.0053 -0.0049
 -0.0049 -0.0027  0.0002  0.0006  0.0013  0.0019  0.0042  0.0049  0.0065
  0.0089  0.0167]
Historical VaR95 (2nd-worst return): $155,000


## 2. Portfolio VaR with the variance-covariance matrix (Section 8.3.1)

In [2]:
cov_matrix = np.array([[0.000414, -0.000198], [-0.000198, 0.000118]])
w = np.array([0.5, 0.5])
port_var = w @ cov_matrix @ w
port_std = np.sqrt(port_var)
print(f"Portfolio std: {port_std:.4%}")

V2 = 2_000_000
VaR95_matrix = z95 * port_std * V2
VaR99_matrix = z99 * port_std * V2
print(f"Matrix-based VaR95: ${VaR95_matrix:,.0f}")
print(f"Matrix-based VaR99: ${VaR99_matrix:,.0f}")

Portfolio std: 0.5831%
Matrix-based VaR95: $19,182
Matrix-based VaR99: $27,130


## 2b. Component VaR: decomposing portfolio risk by position (Section 8.3.2)

In [3]:
Sigma_w = cov_matrix @ w
comp_var = w * Sigma_w
print(f"Sigma @ w: {Sigma_w}")
print(f"Component variance: {comp_var}, sum={comp_var.sum():.6f} (check vs port_var={port_var:.6f})")

frac = comp_var / port_var
comp_VaR = frac * VaR95_matrix
print(f"Fractional contribution: {frac}")
print(f"Component VaR95: AssetA=${comp_VaR[0]:,.0f}, AssetB=${comp_VaR[1]:,.0f}, sum=${comp_VaR.sum():,.0f}")

Sigma @ w: [ 1.08e-04 -4.00e-05]
Component variance: [ 5.4e-05 -2.0e-05], sum=0.000034 (check vs port_var=0.000034)
Fractional contribution: [ 1.58823529 -0.58823529]
Component VaR95: AssetA=$30,466, AssetB=$-11,284, sum=$19,182


## 3. Expected Shortfall (Section 8.2.3)

In [4]:
tail = sorted_returns[:idx95 + 1]
ES95_hist = -tail.mean() * V
print(f"Historical ES95 (average of two worst returns): ${ES95_hist:,.0f}")

ES95_param = (sigma * norm.pdf(z95) / 0.05 - mu) * V
print(f"Parametric ES95: ${ES95_param:,.0f}")

Historical ES95 (average of two worst returns): $188,500
Parametric ES95: $225,366


## 3b. Maximum drawdown (Section 8.2.4)

In [5]:
values = np.concatenate(([V], V * np.cumprod(1 + returns)))
running_peak = np.maximum.accumulate(values)
drawdown = (running_peak - values) / running_peak
mdd_idx = int(np.argmax(drawdown))
peak_idx = int(np.argmax(values[:mdd_idx + 1]))
MDD = drawdown[mdd_idx]
print(f"Running peak: ${values[peak_idx]:,.0f} (after Day {peak_idx})")
print(f"Trough: ${values[mdd_idx]:,.0f} (after Day {mdd_idx})")
print(f"Maximum drawdown: {MDD:.2%}, or ${values[peak_idx] - values[mdd_idx]:,.0f}")
cumulative_return = values[-1] / V - 1
print(f"Cumulative 20-day return: {cumulative_return:.2%}")

Running peak: $10,048,025 (after Day 2)
Trough: $9,371,210 (after Day 20)
Maximum drawdown: 6.74%, or $676,815
Cumulative 20-day return: -6.29%


## 4. Monte Carlo simulation for risk measurement (Section 8.2.5)

In [6]:
rng = np.random.default_rng(42)
sims = rng.normal(mu, sigma, 100_000)
sim_losses = -sims * V

VaR95_mc = np.percentile(sim_losses, 95)
VaR99_mc = np.percentile(sim_losses, 99)
ES95_mc = sim_losses[sim_losses >= VaR95_mc].mean()

print(f"Monte Carlo VaR95: ${VaR95_mc:,.0f}")
print(f"Monte Carlo VaR99: ${VaR99_mc:,.0f}")
print(f"Monte Carlo ES95:  ${ES95_mc:,.0f}")

Monte Carlo VaR95: $187,460
Monte Carlo VaR99: $251,690
Monte Carlo ES95:  $226,378


## 4b. Volatility-adjusted (GARCH-based) VaR (Section 8.2.5)

In [7]:
# GARCH(1,1) forecast volatilities from Chapter 7's recursion (t=3 pre-shock, t=4 post-shock)
sigma_t3, sigma_t4 = 0.0195, 0.0208
V_garch = 10_000_000

VaR95_t3 = z95 * sigma_t3 * V_garch
VaR95_t4 = z95 * sigma_t4 * V_garch
print(f"VaR95 pre-shock (sigma={sigma_t3:.2%}): ${VaR95_t3:,.0f}")
print(f"VaR95 post-shock (sigma={sigma_t4:.2%}): ${VaR95_t4:,.0f}")
print(f"Relative increase: {(VaR95_t4-VaR95_t3)/VaR95_t3:.2%}")

VaR95 pre-shock (sigma=1.95%): $320,746
VaR95 post-shock (sigma=2.08%): $342,130
Relative increase: 6.67%


## 4c. Filtered historical simulation (Section 8.2.5)

In [8]:
# Filtered historical simulation: standardize a historical return by the GARCH
# volatility prevailing when it was observed, then rescale by today's forecast.
# Uses the chapter's stated (rounded) sigmas, so this matches the text exactly.
r_shock_fhs = 0.0297          # Chapter 2's AssetA return, observed at t=3
z3_fhs = r_shock_fhs / sigma_t3
r_scaled_fhs = z3_fhs * sigma_t4

print(f"Standardized residual: z_3 = {r_shock_fhs:.4f} / {sigma_t3:.4f} = {z3_fhs:.3f}")
print(f"Rescaled by today's sigma ({sigma_t4:.2%}): {r_scaled_fhs:.2%}")
print(f"Plain historical simulation would reuse:  {r_shock_fhs:.2%}")
print(f"Understatement avoided: {(r_scaled_fhs - r_shock_fhs) * 10000:.0f} bp")

Standardized residual: z_3 = 0.0297 / 0.0195 = 1.523
Rescaled by today's sigma (2.08%): 3.17%
Plain historical simulation would reuse:  2.97%
Understatement avoided: 20 bp


## 5. Backtesting Value at Risk models (Section 8.3.1)

In [9]:
r_star_95 = mu - z95 * sigma
r_star_99 = mu - z99 * sigma
exceptions_95 = (returns < r_star_95).sum()
exceptions_99 = (returns < r_star_99).sum()

print(f"95% VaR return threshold: {r_star_95:.4%}")
print(f"Observed exceptions at 95%: {exceptions_95} (expected {0.05*len(returns):.1f})")
print(f"99% VaR return threshold: {r_star_99:.4%}")
print(f"Observed exceptions at 99%: {exceptions_99} (expected {0.01*len(returns):.1f})")

95% VaR return threshold: -1.8619%
Observed exceptions at 95%: 1 (expected 1.0)
99% VaR return threshold: -2.5008%
Observed exceptions at 99%: 0 (expected 0.2)


## 5b. Kupiec's proportion-of-failures test (Section 8.3.1)

In [10]:
p, n, x = 0.05, 20, 1
num = (1-p)**(n-x) * p**x
xn = x/n
den = (1-xn)**(n-x) * xn**x
LR_POF = -2*np.log(num/den)
print(f"Observed rate x/n = {xn:.4f} (target p={p})")
print(f"Kupiec LR_POF statistic: {LR_POF:.4f} (chi-sq(1) 95% critical value = 3.841)")

Observed rate x/n = 0.0500 (target p=0.05)
Kupiec LR_POF statistic: -0.0000 (chi-sq(1) 95% critical value = 3.841)


## 5c. Backtesting Expected Shortfall (Section 8.3.1)

In [11]:
exception_loss = -sorted_returns[0] * V  # worst return, the single 95% exception
print(f"Realized loss on exception day: ${exception_loss:,.0f}")
print(f"Parametric ES95 forecast: ${ES95_param:,.0f}")
print(f"Ratio (realized/forecast): {exception_loss/ES95_param:.3f}")

Realized loss on exception day: $222,000
Parametric ES95 forecast: $225,366
Ratio (realized/forecast): 0.985


## 5d. From daily VaR to economic capital: the square-root-of-time rule and its limits (Section 8.3.1)

In [12]:
# Square-root-of-time scaling of the parametric 99% VaR to a 10-day horizon
T_scale = 10
VaR10_sqrtT = VaR99_param * np.sqrt(T_scale)
print(f"1-day VaR99: ${VaR99_param:,.0f}")
print(f"10-day VaR99 (sqrt-T rule): ${VaR10_sqrtT:,.0f}")

# True 10-day variance under a modest AR(1) autocorrelation phi=0.10
phi_ec = 0.10
var_iid = T_scale
var_ar1 = T_scale + 2 * sum((T_scale - k) * phi_ec**k for k in range(1, T_scale))
scaling_factor = np.sqrt(var_ar1 / var_iid)
VaR10_true = VaR10_sqrtT * scaling_factor
print(f"\nWith phi={phi_ec} autocorrelation:")
print(f"True 10-day VaR99: ${VaR10_true:,.0f}")
print(f"Understatement from the sqrt-T rule: {(scaling_factor-1)*100:.2f}%")

1-day VaR99: $250,081
10-day VaR99 (sqrt-T rule): $790,824

With phi=0.1 autocorrelation:
True 10-day VaR99: $865,413
Understatement from the sqrt-T rule: 9.43%


## 6. Liquidity-adjusted VaR and the NSFR (Section 8.3.2)

In [13]:
spread_bps = 30
liquidity_cost = 0.5 * (spread_bps / 10_000) * V2
LVaR95 = VaR95_matrix + liquidity_cost
print(f"Liquidity cost adjustment: ${liquidity_cost:,.0f}")
print(f"Liquidity-adjusted VaR95: ${LVaR95:,.0f}")

# Liquidity Coverage Ratio
hqla = 50_000_000
net_outflows = 40_000_000
lcr = hqla / net_outflows
print(f"\nLiquidity Coverage Ratio: {lcr:.0%}")

Liquidity cost adjustment: $3,000
Liquidity-adjusted VaR95: $22,182

Liquidity Coverage Ratio: 125%


## 6b. VaR for nonlinear portfolios: delta-normal and delta-gamma (Section 8.3.2)

In [14]:
S0, K, r_bs, sigma_bs, T_bs = 50.0, 50.0, 0.04, 0.30, 1.0
d1 = (np.log(S0/K) + (r_bs + sigma_bs**2/2)*T_bs) / (sigma_bs*np.sqrt(T_bs))
d2 = d1 - sigma_bs*np.sqrt(T_bs)
C0 = S0*norm.cdf(d1) - K*np.exp(-r_bs*T_bs)*norm.cdf(d2)
delta = norm.cdf(d1)
gamma = norm.pdf(d1) / (S0*sigma_bs*np.sqrt(T_bs))
print(f"C0={C0:.4f} delta={delta:.4f} gamma={gamma:.4f}")

# The chapter states z = 1.645 / 2.326 for this options example and expects the
# reader to check it by hand, so use those rounded quantiles here rather than
# norm.ppf, which the parametric-VaR section above legitimately uses instead.
z95_opt, z99_opt = 1.645, 2.326
daily_sigma = sigma_bs / np.sqrt(252)
n_contracts = 1000

delta_dollar_stdev = delta * S0 * daily_sigma * n_contracts
VaR95_dn = z95_opt * delta_dollar_stdev
VaR99_dn = z99_opt * delta_dollar_stdev
print(f"\nDelta-normal VaR95: ${VaR95_dn:.2f}, VaR99: ${VaR99_dn:.2f}")

# Full revaluation
S_up_95 = S0*(1+z95_opt*daily_sigma)
S_up_99 = S0*(1+z99_opt*daily_sigma)
def bs_call(S):
    d1_ = (np.log(S/K) + (r_bs + sigma_bs**2/2)*T_bs) / (sigma_bs*np.sqrt(T_bs))
    d2_ = d1_ - sigma_bs*np.sqrt(T_bs)
    return S*norm.cdf(d1_) - K*np.exp(-r_bs*T_bs)*norm.cdf(d2_)

C_up_95, C_up_99 = bs_call(S_up_95), bs_call(S_up_99)
loss_full_95 = (C_up_95-C0)*n_contracts
loss_full_99 = (C_up_99-C0)*n_contracts
print(f"Full-revaluation loss95: ${loss_full_95:.2f}, loss99: ${loss_full_99:.2f}")

# Delta-gamma
dS95, dS99 = S0*z95_opt*daily_sigma, S0*z99_opt*daily_sigma
dg95 = n_contracts*(delta*dS95 + 0.5*gamma*dS95**2)
dg99 = n_contracts*(delta*dS99 + 0.5*gamma*dS99**2)
print(f"Delta-gamma loss95: ${dg95:.2f}, loss99: ${dg99:.2f}")

C0=6.8766 delta=0.6115 gamma=0.0255

Delta-normal VaR95: $950.56, VaR99: $1344.08
Full-revaluation loss95: $980.79, loss99: $1403.99
Delta-gamma loss95: $981.43, loss99: $1405.79


## 7. Sensitivity-based risk limits: the full Greek profile (Section 8.4.1)

In [15]:
# Section 8.4.1: the full Greek profile of the short 1,000-call position.
# delta, gamma, n_contracts and the option parameters all come from Section 8.3.2 above,
# so the Greek table cannot drift from the VaR figures computed there.
grk_vega = S0 * norm.pdf(d1) * np.sqrt(T_bs) / 100            # per percentage point of vol
grk_theta = (-(S0 * norm.pdf(d1) * sigma_bs) / (2 * np.sqrt(T_bs))
             - r_bs * K * np.exp(-r_bs * T_bs) * norm.cdf(d2)) / 365    # per day
grk_rho = K * T_bs * np.exp(-r_bs * T_bs) * norm.cdf(d2) / 100          # per percentage point

print("per option:")
print(f"  delta {delta:.6f}  gamma {gamma:.6f}  vega {grk_vega:.6f}"
      f"  theta {grk_theta:.6f}  rho {grk_rho:.6f}")

print(f"\nposition, short {n_contracts} contracts:")
for nm, val, unit in (("delta", -n_contracts * delta, "shares"),
                      ("gamma", -n_contracts * gamma, "per $1 move"),
                      ("vega", -n_contracts * grk_vega, "per +1 vol point"),
                      ("theta", -n_contracts * grk_theta, "per day"),
                      ("rho", -n_contracts * grk_rho, "per +1 rate point")):
    print(f"  {nm:<6}{val:>12,.2f}  {unit}")

# Every VaR figure in Section 8.3.2 shocks the stock price and holds vol fixed,
# so vega risk is structurally invisible to it.
grk_vol_cost = n_contracts * grk_vega * 5
print(f"\n+5 vol points costs ${grk_vol_cost:,.2f}, "
      f"{grk_vol_cost / loss_full_99:.1%} of the ${loss_full_99:,.2f} 99% full-revaluation VaR")

per option:
  delta 0.611539  gamma 0.025550  vega 0.191623  theta -0.010472  rho 0.237003

position, short 1000 contracts:
  delta      -611.54  shares
  gamma       -25.55  per $1 move
  vega       -191.62  per +1 vol point
  theta        10.47  per day
  rho        -237.00  per +1 rate point

+5 vol points costs $958.12, 68.2% of the $1,403.99 99% full-revaluation VaR


## 7b. The bucketed DV01 ladder (Section 8.3.3)

In [16]:
# Section 8.3.3: a bucketed DV01 ladder. Each figure is the position's P&L
# from a one-basis-point RISE in that bucket's yield, so a long bond is negative.
dv01_buckets = {"2y": -25_000, "5y": -40_000, "10y": 38_000, "30y": 27_000}
dv01_parallel = {b: 10 for b in dv01_buckets}
dv01_flattening = {"2y": 20, "5y": 10, "10y": -5, "30y": -10}
dv01_steepening = {"2y": -10, "5y": -5, "10y": 10, "30y": 15}

print(f"net DV01   {sum(dv01_buckets.values()):>10,} per bp")
print(f"gross DV01 {sum(abs(v) for v in dv01_buckets.values()):>10,} per bp\n")

for dv01_name, dv01_scen in (("parallel +10bp", dv01_parallel),
                             ("flattening", dv01_flattening),
                             ("steepening", dv01_steepening)):
    dv01_pnl = sum(dv01_buckets[b] * dv01_scen[b] for b in dv01_buckets)
    print(f"{dv01_name:>16}: {dv01_pnl:>12,.0f}")

net DV01            0 per bp
gross DV01    130,000 per bp

  parallel +10bp:            0
      flattening:   -1,360,000
      steepening:    1,235,000


## 7c. Adjoint differentiation: every Greek in one reverse sweep (Section 8.3.3)

In [17]:
# Section 8.3.3: adjoint (reverse-mode) differentiation of Black-Scholes.
# This is backpropagation: record a tape on the forward pass, then push
# derivatives back through it ONCE to recover every sensitivity at the same time.
import math
from statistics import NormalDist

ad_Phi, ad_phi = NormalDist().cdf, NormalDist().pdf


class ADVar:
    """A node on the tape: a value, plus how it was built from its parents."""

    def __init__(self, value, parents=()):
        self.value, self.parents, self.grad = float(value), parents, 0.0

    def __add__(self, o):
        o = o if isinstance(o, ADVar) else ADVar(o)
        return ADVar(self.value + o.value, ((self, 1.0), (o, 1.0)))

    def __sub__(self, o):
        o = o if isinstance(o, ADVar) else ADVar(o)
        return ADVar(self.value - o.value, ((self, 1.0), (o, -1.0)))

    def __mul__(self, o):
        o = o if isinstance(o, ADVar) else ADVar(o)
        return ADVar(self.value * o.value, ((self, o.value), (o, self.value)))

    def __truediv__(self, o):
        o = o if isinstance(o, ADVar) else ADVar(o)
        return ADVar(self.value / o.value,
                     ((self, 1.0 / o.value), (o, -self.value / o.value ** 2)))

    __radd__, __rmul__ = __add__, __mul__


def _ad_unary(x, value, local):
    return ADVar(value, ((x, local),))


def ad_exp(x):
    return _ad_unary(x, math.exp(x.value), math.exp(x.value))


def ad_log(x):
    return _ad_unary(x, math.log(x.value), 1.0 / x.value)


def ad_sqrt(x):
    return _ad_unary(x, math.sqrt(x.value), 0.5 / math.sqrt(x.value))


def ad_cdf(x):
    return _ad_unary(x, ad_Phi(x.value), ad_phi(x.value))   # dN/dx is the normal pdf


def ad_backward(node):
    """One reverse sweep: seed the output with 1, apply the chain rule backward."""
    ad_order, ad_seen = [], set()

    def build(n):
        if id(n) in ad_seen:
            return
        ad_seen.add(id(n))
        for p, _ in n.parents:
            build(p)
        ad_order.append(n)

    build(node)
    for n in ad_order:
        n.grad = 0.0
    node.grad = 1.0
    for n in reversed(ad_order):
        for p, local in n.parents:
            p.grad += n.grad * local


def ad_price(S, K_, r_, sig_, T_):
    sqrtT = ad_sqrt(T_)
    ad_d1 = (ad_log(S / K_) + (r_ + sig_ * sig_ * 0.5) * T_) / (sig_ * sqrtT)
    ad_d2 = ad_d1 - sig_ * sqrtT
    return S * ad_cdf(ad_d1) - K_ * ad_exp(ADVar(0.0) - r_ * T_) * ad_cdf(ad_d2)


ad_S, ad_K = ADVar(S0), ADVar(K)
ad_r, ad_sig, ad_T = ADVar(r_bs), ADVar(sigma_bs), ADVar(T_bs)
ad_C = ad_price(ad_S, ad_K, ad_r, ad_sig, ad_T)
ad_backward(ad_C)

print(f"price {ad_C.value:.6f}  (closed form {C0:.6f})\n")
print("ONE reverse sweep returns all four first-order Greeks:")
print(f"  delta = dC/dS         {ad_S.grad:.9f}   closed form {delta:.9f}")
print(f"  vega  = dC/dsigma/100 {ad_sig.grad / 100:.9f}   closed form {grk_vega:.9f}")
print(f"  theta = -dC/dT/365    {-ad_T.grad / 365:.9f}   closed form {grk_theta:.9f}")
print(f"  rho   = dC/dr/100     {ad_r.grad / 100:.9f}   closed form {grk_rho:.9f}")


def ad_delta_at(s0):
    """Gamma is second order, so the machinery has to be applied to its own output."""
    v = ADVar(s0)
    ad_backward(ad_price(v, ADVar(K), ADVar(r_bs), ADVar(sigma_bs), ADVar(T_bs)))
    return v.grad


ad_h = 1e-4
ad_gamma = (ad_delta_at(S0 + ad_h) - ad_delta_at(S0 - ad_h)) / (2 * ad_h)
print(f"\n  gamma (nested, off the AD delta) {ad_gamma:.9f}   closed form {gamma:.9f}")

price 6.876632  (closed form 6.876632)

ONE reverse sweep returns all four first-order Greeks:
  delta = dC/dS         0.611539336   closed form 0.611539336
  vega  = dC/dsigma/100 0.191623149   closed form 0.191623149
  theta = -dC/dT/365    -0.010472221   closed form -0.010472221
  rho   = dC/dr/100     0.237003345   closed form 0.237003345

  gamma (nested, off the AD delta) 0.025549753   closed form 0.025549753


## 8. Stress testing and correlation stress testing (Section 8.4.1)

In [18]:
stress_return = -0.07
stress_loss = -stress_return * V
print(f"Stressed loss (single-day -7% return): ${stress_loss:,.0f}")
print(f"Compare to parametric VaR99: ${VaR99_param:,.0f}")

# Correlation stress test
sigA, sigB = np.sqrt(cov_matrix[0,0]), np.sqrt(cov_matrix[1,1])
rho_calm = cov_matrix[0,1] / (sigA*sigB)
print(f"\nsigA={sigA:.4%} sigB={sigB:.4%} rho_calm={rho_calm:.4f}")

rho_stress = 0.5
cov_stress = rho_stress * sigA * sigB
Sigma_stress = np.array([[cov_matrix[0,0], cov_stress],[cov_stress, cov_matrix[1,1]]])
port_var_stress = w @ Sigma_stress @ w
port_std_stress = np.sqrt(port_var_stress)
print(f"Stressed portfolio std: {port_std_stress:.4%}")

VaR95_stress = z95 * port_std_stress * V2
print(f"VaR95 stress: ${VaR95_stress:,.0f}  (ratio to calm VaR95={VaR95_stress/VaR95_matrix:.2f}x)")

Stressed loss (single-day -7% return): $700,000
Compare to parametric VaR99: $250,081

sigA=2.0347% sigB=1.0863% rho_calm=-0.8958
Stressed portfolio std: 1.3721%
VaR95 stress: $45,137  (ratio to calm VaR95=2.35x)


## 8b. Reverse stress testing and a VaR limit breach (Section 8.4.1, Section 8.5.3)

Reverse stress test: what single-day return exhausts a $500,000 risk capital cushion? VaR limit example: utilization of a $300,000 approved limit, and a subsequent breach.

In [19]:
# Reverse stress test
cushion = 500_000
reverse_stress_return = cushion / V
print(f"Return that exhausts a ${cushion:,.0f} cushion: {reverse_stress_return:.2%}")
print(f"Historical -7% stress loss (${stress_loss:,.0f}) exceeds cushion by "
      f"${stress_loss - cushion:,.0f}")

# VaR limit utilization and breach
approved_limit = 300_000
utilization = VaR99_param / approved_limit
print(f"\nVaR99 limit utilization: {utilization:.1%}")

breached_var = 340_000
breach_amount = breached_var - approved_limit
print(f"Breach: ${breach_amount:,.0f} ({breach_amount/approved_limit:.1%} over limit)")

Return that exhausts a $500,000 cushion: 5.00%
Historical -7% stress loss ($700,000) exceeds cushion by $200,000

VaR99 limit utilization: 83.4%
Breach: $40,000 (13.3% over limit)


## 9. Machine learning for volatility forecasting: GARCH(1,1) vs. gradient boosting (Section 8.4.2)

Chapter 7 noted that a plain GARCH(1,1) model treats a positive and a negative shock of the same size as equally informative about future volatility, a well-documented shortcoming called the leverage effect. Here we test whether a gradient-boosted tree, given explicit access to the sign of each lagged return (information a symmetric GARCH(1,1) cannot use), can out-forecast GARCH(1,1) on a return series simulated from a *known* asymmetric (GJR-GARCH-style) process, where negative shocks raise variance more than equally-sized positive ones.

In [20]:
from arch import arch_model
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Simulate 750 days from a known asymmetric (GJR-GARCH-style) process:
# negative shocks get an extra variance kick (gamma) that positive shocks do not.
rng_vol = np.random.default_rng(7)
n_days_vol = 750
omega_true, alpha_true, beta_true, gamma_true = 0.000005, 0.05, 0.85, 0.12

sigma2_true = np.zeros(n_days_vol)
r_sim = np.zeros(n_days_vol)
sigma2_true[0] = omega_true / (1 - alpha_true - beta_true - gamma_true / 2)
r_sim[0] = rng_vol.normal(0, np.sqrt(sigma2_true[0]))
for t in range(1, n_days_vol):
    shock = r_sim[t - 1]
    leverage_term = gamma_true * shock**2 * (1.0 if shock < 0 else 0.0)
    sigma2_true[t] = omega_true + alpha_true * shock**2 + leverage_term + beta_true * sigma2_true[t - 1]
    r_sim[t] = rng_vol.normal(0, np.sqrt(sigma2_true[t]))

n_train_vol = 600
r_train, r_test = r_sim[:n_train_vol], r_sim[n_train_vol:]
sigma2_test_true = sigma2_true[n_train_vol:]
print(f"Simulated {n_days_vol} days ({n_train_vol} train, {n_days_vol - n_train_vol} test)")

Simulated 750 days (600 train, 150 test)


In [21]:
# Fit a plain, symmetric GARCH(1,1) via maximum likelihood -- structurally
# misspecified for this asymmetric process, since it cannot see the shock's sign.
am = arch_model(r_train * 100, mean="Zero", vol="GARCH", p=1, q=1, dist="normal")
res = am.fit(disp="off")
omega_hat = res.params["omega"] / 100**2
alpha_hat = res.params["alpha[1]"]
beta_hat = res.params["beta[1]"]
print(f"GARCH(1,1) fitted: omega={omega_hat:.3e}, alpha={alpha_hat:.4f}, beta={beta_hat:.4f}")

# One-step-ahead walk-forward forecasts over the test period
last_sigma2 = res.conditional_volatility[-1]**2 / 100**2
last_r = r_train[-1]
garch_forecasts = []
for t in range(len(r_test)):
    next_sigma2 = omega_hat + alpha_hat * last_r**2 + beta_hat * last_sigma2
    garch_forecasts.append(next_sigma2)
    last_sigma2 = next_sigma2
    last_r = r_test[t]
garch_forecasts = np.array(garch_forecasts)
garch_rmse = np.sqrt(np.mean((garch_forecasts - sigma2_test_true)**2))
print(f"GARCH(1,1) forecast-variance RMSE vs. true simulated variance: {garch_rmse:.3e}")

GARCH(1,1) fitted: omega=7.866e-06, alpha=0.1597, beta=0.7641
GARCH(1,1) forecast-variance RMSE vs. true simulated variance: 2.416e-05


In [22]:
# Gradient-boosted regressor with sign-aware features (lagged squared AND signed
# returns, so the model *can* in principle learn the leverage asymmetry GARCH(1,1)
# cannot represent), hyperparameters chosen by time-series-aware cross-validation.
sq_sim = r_sim**2
X_vol, y_vol = [], []
for t in range(2, n_train_vol):
    X_vol.append([sq_sim[t - 1], sq_sim[t - 2], r_sim[t - 1], r_sim[t - 2]])
    y_vol.append(sq_sim[t])
X_vol, y_vol = np.array(X_vol), np.array(y_vol)

param_grid = {"n_estimators": [10, 25, 50, 100], "max_depth": [1, 2, 3], "learning_rate": [0.01, 0.05, 0.1]}
tscv = TimeSeriesSplit(n_splits=5)
gs = GridSearchCV(GradientBoostingRegressor(random_state=0), param_grid, cv=tscv, scoring="neg_mean_squared_error")
gs.fit(X_vol, y_vol)
gbm_vol = gs.best_estimator_
print(f"Best GBM hyperparameters: {gs.best_params_}")

gbm_forecasts = np.array([
    gbm_vol.predict([[sq_sim[t - 1], sq_sim[t - 2], r_sim[t - 1], r_sim[t - 2]]])[0]
    for t in range(n_train_vol, n_days_vol)
])
gbm_rmse = np.sqrt(np.mean((gbm_forecasts - sigma2_test_true)**2))
print(f"Gradient boosting forecast-variance RMSE vs. true simulated variance: {gbm_rmse:.3e}")

naive_rmse = np.sqrt(np.mean((np.var(r_train) - sigma2_test_true)**2))
print(f"Naive (constant unconditional variance) RMSE vs. true simulated variance: {naive_rmse:.3e}")

print(f"\nGARCH(1,1) beats gradient boosting by {(1 - garch_rmse/gbm_rmse)*100:.1f}%")
print(f"Gradient boosting beats the naive benchmark by {(1 - gbm_rmse/naive_rmse)*100:.1f}%")

Best GBM hyperparameters: {'learning_rate': 0.05, 'max_depth': 1, 'n_estimators': 100}
Gradient boosting forecast-variance RMSE vs. true simulated variance: 3.830e-05
Naive (constant unconditional variance) RMSE vs. true simulated variance: 4.202e-05

GARCH(1,1) beats gradient boosting by 36.9%
Gradient boosting beats the naive benchmark by 8.8%


## 10. Managing the risk: hedging, position sizing, drawdown triggers (Section 8.5.2)

In [23]:
# Section 8.6: the three mitigation controls, on this chapter's own numbers.

# 8.13.1 -- delta hedging removes the linear risk and leaves gamma
mgmt_shares = n_contracts * delta
# The chapter quotes the rounded constants dS_99 = $2.198 and sigma = 0.937% and
# expects the reader to check them by hand, so reuse those displayed values.
mgmt_dS99, mgmt_sigma = 2.198, 0.00937
mgmt_residual = 0.5 * n_contracts * gamma * mgmt_dS99 ** 2
print("hedging")
print(f"  buy {mgmt_shares:,.1f} shares to flatten delta")
print(f"  residual at the 99% move: ${mgmt_residual:,.2f} vs ${loss_full_99:,.2f} unhedged"
      f"  ({1 - mgmt_residual / loss_full_99:.1%} reduction)")
print(f"  double the move -> ${0.5 * n_contracts * gamma * (2 * mgmt_dS99) ** 2:,.2f} (quadratic, and a")
print(f"  loss for a move in EITHER direction, since the sign of dS is squared away)")

# 8.13.2 -- volatility targeting sizes the position before it is taken
mgmt_ann_vol = mgmt_sigma * np.sqrt(252)
mgmt_target = 0.10
mgmt_scale = mgmt_target / mgmt_ann_vol
print("\nposition sizing")
print(f"  daily sigma {mgmt_sigma:.4%} -> annualized {mgmt_ann_vol:.4%}")
print(f"  scale to a {mgmt_target:.0%} target: {mgmt_scale:.4f} -> ${V * mgmt_scale:,.0f} of ${V:,.0f}")

# 8.13.3 -- a 5% drawdown trigger on the same twenty-day path
mgmt_values = np.concatenate(([V], V * np.cumprod(1 + returns)))
mgmt_dd = mgmt_values / np.maximum.accumulate(mgmt_values) - 1
mgmt_trigger = int(np.argmax(mgmt_dd <= -0.05))
print("\nstop-loss")
print(f"  maximum drawdown {mgmt_dd.min():.3%}, reached on day {int(np.argmin(mgmt_dd))}")
print(f"  a -5% trigger first fires on day {mgmt_trigger} (drawdown {mgmt_dd[mgmt_trigger]:.3%})")
print(f"  liquidating there avoids only the final day's {returns[-1]:.2%}, "
      f"about {abs(returns[-1]) * 10000:.0f}bp of a {abs(mgmt_dd.min()) * 10000:.0f}bp decline")

hedging
  buy 611.5 shares to flatten delta
  residual at the 99% move: $61.72 vs $1,403.99 unhedged  (95.6% reduction)
  double the move -> $246.87 (quadratic, and a
  loss for a move in EITHER direction, since the sign of dS is squared away)

position sizing
  daily sigma 0.9370% -> annualized 14.8744%
  scale to a 10% target: 0.6723 -> $6,722,954 of $10,000,000

stop-loss
  maximum drawdown -6.736%, reached on day 20
  a -5% trigger first fires on day 19 (drawdown -5.325%)
  liquidating there avoids only the final day's -1.49%, about 149bp of a 674bp decline


## Exercises (match Chapter 8, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [24]:
# Exercise 2: parametric 90% VaR
z90 = norm.ppf(0.90)
VaR90_param = -(mu - z90 * sigma) * V
print(f"Exercise 2 -- Parametric VaR90: ${VaR90_param:,.0f}")

# Exercise 8: Basel traffic-light classification, 7 exceptions in 250 days at 99%
exceptions_250 = 7
zone = "green" if exceptions_250 <= 4 else ("yellow" if exceptions_250 <= 9 else "red")
print(f"Exercise 8 -- {exceptions_250} exceptions in 250 days -> {zone} zone")

# Exercise 12: stressed correlation rho=0.8
rho_ex12 = 0.8
cov_ex12 = rho_ex12*sigA*sigB
Sigma_ex12 = np.array([[cov_matrix[0,0], cov_ex12],[cov_ex12, cov_matrix[1,1]]])
port_std_ex12 = np.sqrt(w @ Sigma_ex12 @ w)
VaR95_ex12 = z95*port_std_ex12*V2
print(f"Exercise 12 -- rho=0.8: portfolio std={port_std_ex12:.4%}, VaR95=${VaR95_ex12:,.0f}")

# Exercise 13: LCR
hqla_ex13, outflows_ex13 = 60_000_000, 55_000_000
print(f"Exercise 13 -- LCR: {hqla_ex13/outflows_ex13:.1%}")

# Exercise (limit utilization): tighter $275,000 limit
tighter_limit = 275_000
util_tighter = VaR99_param / tighter_limit
print(f"\nExercise (limit) -- utilization at $275,000 limit: {util_tighter:.1%}")
print(f"Would $290,000 breach this limit? {290_000 > tighter_limit}")

# Exercise (reverse stress test): $800,000 cushion
cushion_ex = 800_000
reverse_return_ex = cushion_ex / V
print(f"\nExercise (reverse stress) -- return exhausting $800,000 cushion: {reverse_return_ex:.2%}")
print(f"Does the -7% stress loss (${stress_loss:,.0f}) exceed this cushion? {stress_loss > cushion_ex}")

Exercise 2 -- Parametric VaR90: $152,137
Exercise 8 -- 7 exceptions in 250 days -> yellow zone
Exercise 12 -- rho=0.8: portfolio std=1.4880%, VaR95=$48,950
Exercise 13 -- LCR: 109.1%

Exercise (limit) -- utilization at $275,000 limit: 90.9%
Would $290,000 breach this limit? True

Exercise (reverse stress) -- return exhausting $800,000 cushion: 8.00%
Does the -7% stress loss ($700,000) exceed this cushion? False
